# 📚 LightGCN+ v4 — Final Model
### STAT-542 Final Project

**All improvements over vanilla LightGCN:**

*Dataset improvements:*
- Rating-weighted graph edges (rating/10 — strong preferences propagate more)
- Implicit feedback as auxiliary graph edges (weight=0.1, not in BPR loss)
- Author-book edges in graph (books by same author share signal directly)
- City/state region features for users in large countries

*Architectural improvements (LightGCN itself):*
- Residual skip connections E^(k) = A_hat @ E^(k-1) + alpha * E^(0) — prevents over-smoothing
- Adaptive layer dropout — deeper layers get more dropout (noisier on sparse data)
- Separate learnable temperature for cold vs warm users
- L2 normalized embeddings before scoring (cosine similarity — fixes magnitude bias)

*Training improvements:*
- Rating-weighted BPR loss (rating-10 triplets contribute more than rating-6)
- Hard negative mining after epoch 20 (semi-hard negatives keep gradients informative)
- Popularity-weighted negatives (epochs 1-20)

*Cold start:*
- Two-regime model: cold users (≤10 ratings) use features only
- Warm users use full graph propagation with residual connections

*Evaluation:*
- 99-sampled negatives (comparable to NeuMF)
- Full ranking evaluation (more honest)


## Step 1 — Dependencies

In [ ]:
import subprocess, sys
pkgs = ['torch','numpy','pandas','scipy','scikit-learn','matplotlib','seaborn']
subprocess.check_call([sys.executable,'-m','pip','install','--quiet']+pkgs)
print('All dependencies ready.')


## Step 2 — Device

In [ ]:
import torch, os
def get_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
        torch.set_num_threads(10)
        print('Device: Apple MPS — M4 Pro')
        return torch.device('mps')
    elif torch.cuda.is_available():
        print(f'Device: CUDA — {torch.cuda.get_device_name(0)}')
        return torch.device('cuda')
    torch.set_num_threads(os.cpu_count() or 4)
    print(f'Device: CPU ({os.cpu_count()} threads)')
    return torch.device('cpu')
DEVICE = get_device()


## Step 3 — Configuration

In [ ]:
import os, numpy as np, torch
from collections import defaultdict

DATA_DIR         = '/Users/tanmayshikhare/Downloads'
RESULTS_DIR      = 'results_v4'

# Data
MIN_USER_RATINGS = 5
MIN_BOOK_RATINGS = 5
COLD_THRESHOLD   = 10
IMPLICIT_WEIGHT  = 0.1
AUTHOR_EDGE_WEIGHT = 0.3   # weight for author-book edges in graph

# Model
EMBEDDING_DIM    = 256
N_LAYERS         = 4
DROPOUT          = 0.1
RESIDUAL_ALPHA   = 0.2     # residual skip connection strength

# Training
LR               = 1e-3
REG_LAMBDA       = 1e-4
N_EPOCHS         = 100
BATCH_SIZE       = 8192
EVAL_EVERY       = 5
PATIENCE         = 15
RANDOM_SEED      = 42
HARD_NEG_START   = 20      # epoch after which switch to semi-hard negatives

# Evaluation
TOP_K            = [5, 10, 20]
N_NEG_EVAL       = 99

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, 'error_analysis'), exist_ok=True)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f'Results → {os.path.abspath(RESULTS_DIR)}/')
print(f'Cold threshold: {COLD_THRESHOLD} | Implicit weight: {IMPLICIT_WEIGHT}')
print(f'Residual alpha: {RESIDUAL_ALPHA} | Hard neg start: epoch {HARD_NEG_START}')


## Step 4 — Load all 3 dataset files

In [ ]:
import pandas as pd

def load_bookcrossing(data_dir, min_user_ratings=5, min_book_ratings=5):
    print('='*60)
    print('Loading Book-Crossing dataset (all 3 files)')
    print('='*60)

    # FILE 1: Ratings
    print('\n[1/3] BX-Book-Ratings.csv')
    all_ratings = pd.read_csv(os.path.join(data_dir,'BX-Book-Ratings.csv'),
        sep=';', encoding='latin-1', on_bad_lines='skip')
    all_ratings.columns = ['user_id','isbn','rating']
    print(f'  Raw total: {len(all_ratings):,}')
    implicit = all_ratings[all_ratings['rating'] == 0].copy()
    explicit = all_ratings[all_ratings['rating'] >  0].copy()
    explicit['rating'] = explicit['rating'].clip(1,10).astype(np.float32)
    print(f'  Explicit: {len(explicit):,} | Implicit: {len(implicit):,}')

    # Co-filter to convergence on explicit only
    ratings = explicit.copy()
    prev_len, itr = -1, 0
    while len(ratings) != prev_len:
        prev_len = len(ratings)
        uc = ratings['user_id'].value_counts()
        bc = ratings['isbn'].value_counts()
        ratings = ratings[ratings['user_id'].isin(uc[uc >= min_user_ratings].index)]
        ratings = ratings[ratings['isbn'].isin(bc[bc >= min_book_ratings].index)]
        itr += 1
    print(f'  After co-filtering ({itr} passes): {len(ratings):,} explicit ratings')

    active_users = set(ratings['user_id'].unique())
    active_isbns = set(ratings['isbn'].unique())

    # Filter implicit to active users/books only
    implicit_filtered = implicit[
        implicit['user_id'].isin(active_users) &
        implicit['isbn'].isin(active_isbns)].copy()
    print(f'  Implicit for active users/books: {len(implicit_filtered):,}')

    # FILE 2: Users
    print('\n[2/3] BX-Users.csv')
    users_df = pd.read_csv(os.path.join(data_dir,'BX-Users.csv'),
        sep=';', encoding='latin-1', on_bad_lines='skip')
    users_df.columns = ['user_id','location','age']
    users_df['age'] = pd.to_numeric(users_df['age'], errors='coerce')
    users_df.loc[~users_df['age'].between(5,100),'age'] = np.nan
    users_df['age'] = users_df['age'].fillna(users_df['age'].median())
    users_df['age_bin'] = pd.cut(users_df['age'],
        bins=[0,18,25,35,55,120], labels=[0,1,2,3,4]).astype(int)

    # Location: city, state, country
    loc_parts = users_df['location'].str.split(',', expand=True)
    n_parts   = loc_parts.shape[1]
    users_df['country'] = loc_parts[n_parts-1].str.strip().str.lower().fillna('unknown')
    users_df['state']   = loc_parts[1].str.strip().str.lower().fillna('unknown') if n_parts > 1 else 'unknown'

    users_df = users_df[users_df['user_id'].isin(active_users)].copy()
    top_countries = users_df['country'].value_counts().head(50).index
    users_df['country_enc'] = users_df['country'].apply(
        lambda x: x if x in top_countries else 'other')
    country_map = {c:i for i,c in enumerate(users_df['country_enc'].unique())}
    users_df['country_idx'] = users_df['country_enc'].map(country_map)

    # Region: state for top countries, 'other' elsewhere
    top_country_set = {'usa','united states','u.s.a.','uk','united kingdom',
                       'germany','deutschland','canada','australia'}
    users_df['region'] = users_df.apply(
        lambda r: r['state'] if r['country'] in top_country_set
                  and r['state'] not in ('unknown','') else 'other', axis=1)
    top_regions = users_df['region'].value_counts().head(100).index
    users_df['region_enc'] = users_df['region'].apply(
        lambda x: x if x in top_regions else 'other')
    region_map = {r:i for i,r in enumerate(users_df['region_enc'].unique())}
    users_df['region_idx'] = users_df['region_enc'].map(region_map)
    print(f'  Active users: {len(users_df):,} | Countries: {len(country_map)} | Regions: {len(region_map)}')

    # FILE 3: Books
    print('\n[3/3] BX_Books.csv')
    books_df = pd.read_csv(os.path.join(data_dir,'BX_Books.csv'),
        sep=';', encoding='latin-1', on_bad_lines='skip', usecols=[0,1,2,3,4])
    books_df.columns = ['isbn','title','author','year','publisher']
    books_df['year'] = pd.to_numeric(books_df['year'], errors='coerce')
    books_df.loc[~books_df['year'].between(1800,2004),'year'] = np.nan
    books_df['year'] = books_df['year'].fillna(books_df['year'].median())
    books_df['decade'] = ((books_df['year']-1800)//10).clip(0,20).astype(int)
    books_df['author'] = books_df['author'].str.lower().str.strip().fillna('unknown')
    books_df['publisher'] = books_df['publisher'].str.lower().str.strip().fillna('unknown')
    top_authors = books_df['author'].value_counts().head(1000).index
    books_df['author_enc'] = books_df['author'].apply(
        lambda x: x if x in top_authors else 'other')
    author_map = {a:i for i,a in enumerate(books_df['author_enc'].unique())}
    books_df['author_idx'] = books_df['author_enc'].map(author_map)
    top_pubs = books_df['publisher'].value_counts().head(500).index
    books_df['pub_enc'] = books_df['publisher'].apply(
        lambda x: x if x in top_pubs else 'other')
    pub_map = {p:i for i,p in enumerate(books_df['pub_enc'].unique())}
    books_df['pub_idx'] = books_df['pub_enc'].map(pub_map)
    books_df = books_df[books_df['isbn'].isin(active_isbns)].copy()
    print(f'  Active books: {len(books_df):,} | Authors: {len(author_map)} | Publishers: {len(pub_map)}')

    # Re-index
    user_map = {u:i for i,u in enumerate(sorted(ratings['user_id'].unique()))}
    book_map = {b:i for i,b in enumerate(sorted(ratings['isbn'].unique()))}
    ratings['user_idx'] = ratings['user_id'].map(user_map)
    ratings['book_idx'] = ratings['isbn'].map(book_map)
    implicit_filtered['user_idx'] = implicit_filtered['user_id'].map(user_map)
    implicit_filtered['book_idx'] = implicit_filtered['isbn'].map(book_map)
    implicit_filtered = implicit_filtered.dropna(subset=['user_idx','book_idx'])
    implicit_filtered['user_idx'] = implicit_filtered['user_idx'].astype(int)
    implicit_filtered['book_idx'] = implicit_filtered['book_idx'].astype(int)
    books_df['book_idx'] = books_df['isbn'].map(book_map)
    books_df = books_df.dropna(subset=['book_idx'])
    books_df['book_idx'] = books_df['book_idx'].astype(int)

    n_users, n_books = len(user_map), len(book_map)
    sparsity = 100*len(ratings)/(n_users*n_books)
    print(f'\n  ── Dataset Summary ──')
    print(f'  Users:            {n_users:,}')
    print(f'  Books:            {n_books:,}')
    print(f'  Explicit ratings: {len(ratings):,}')
    print(f'  Implicit (aux):   {len(implicit_filtered):,}')
    print(f'  Sparsity:         {sparsity:.5f}%')
    print(f'  Avg/user:         {len(ratings)/n_users:.1f} | Avg/book: {len(ratings)/n_books:.1f}')

    # Feature matrices
    # USER: [age_bin, country_idx, region_idx] — (n_users, 3)
    user_feat = np.zeros((n_users,3), dtype=np.int64)
    u_idx = users_df.copy()
    u_idx['user_idx'] = u_idx['user_id'].map(user_map)
    u_idx = u_idx.dropna(subset=['user_idx'])
    u_idx['user_idx'] = u_idx['user_idx'].astype(int)
    user_feat[u_idx['user_idx'].values,0] = u_idx['age_bin'].values
    user_feat[u_idx['user_idx'].values,1] = u_idx['country_idx'].values
    user_feat[u_idx['user_idx'].values,2] = u_idx['region_idx'].values

    # BOOK: [author_idx, pub_idx, decade] — (n_books, 3)
    book_feat = np.zeros((n_books,3), dtype=np.int64)
    book_feat[books_df['book_idx'].values,0] = books_df['author_idx'].values
    book_feat[books_df['book_idx'].values,1] = books_df['pub_idx'].values
    book_feat[books_df['book_idx'].values,2] = books_df['decade'].values

    meta = {
        'user_map':user_map, 'book_map':book_map,
        'author_map':author_map, 'pub_map':pub_map,
        'country_map':country_map, 'region_map':region_map,
        'books_meta':books_df.set_index('isbn'),
        'books_df': books_df,   # needed for author-book edges
        'n_authors':len(author_map), 'n_pubs':len(pub_map),
        'n_countries':len(country_map), 'n_regions':len(region_map),
        'n_decades':21, 'n_age_bins':5,
    }
    return ratings[['user_idx','book_idx','rating']].copy(), implicit_filtered, n_users, n_books, user_feat, book_feat, meta

df, implicit_df, n_users, n_books, user_feat, book_feat, meta = load_bookcrossing(
    DATA_DIR, MIN_USER_RATINGS, MIN_BOOK_RATINGS)


## Step 5 — Train / Val / Test split

In [ ]:
def split_data(df, test_ratio=0.1, val_ratio=0.1, seed=42):
    train_list, val_list, test_list = [], [], []
    for user, group in df.groupby('user_idx'):
        group = group.sample(frac=1, random_state=seed)
        n = len(group)
        if n < 3:
            train_list.append(group); continue
        n_test = max(1, int(n*test_ratio))
        n_val  = max(1, int(n*val_ratio))
        test_list.append(group.iloc[:n_test])
        val_list.append(group.iloc[n_test:n_test+n_val])
        train_list.append(group.iloc[n_test+n_val:])
    train_df = pd.concat(train_list).reset_index(drop=True)
    val_df   = pd.concat(val_list).reset_index(drop=True)
    test_df  = pd.concat(test_list).reset_index(drop=True)
    print(f'Split — Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
    return train_df, val_df, test_df

train_df, val_df, test_df = split_data(df, 0.1, 0.1, RANDOM_SEED)

train_mask = defaultdict(set)
for u,b in zip(train_df['user_idx'].values, train_df['book_idx'].values):
    train_mask[u].add(int(b))

all_pos = defaultdict(set)
for frame in [train_df, val_df, test_df]:
    for u,b in zip(frame['user_idx'].values, frame['book_idx'].values):
        all_pos[u].add(int(b))

user_train_counts = train_df.groupby('user_idx').size().to_dict()
cold_users = {u for u,c in user_train_counts.items() if c <= COLD_THRESHOLD}
warm_users = {u for u,c in user_train_counts.items() if c  > COLD_THRESHOLD}
print(f'Cold users: {len(cold_users):,} | Warm users: {len(warm_users):,}')


## Step 6 — Build fully augmented graph

In [ ]:
from scipy.sparse import csr_matrix, diags, lil_matrix

def build_norm_adjacency(train_df, implicit_df, books_df,
                         n_users, n_books,
                         implicit_weight=0.1,
                         author_edge_weight=0.3):
    """
    Full augmented adjacency with 3 edge types:

    1. Explicit user-book edges — weight = rating/10
       Strong preferences propagate more signal than weak ones.

    2. Implicit user-book edges — weight = implicit_weight (0.1)
       Weak positive signal from browse/click interactions.
       Enriches cold user neighborhoods without corrupting BPR loss.

    3. Author-book edges — weight = author_edge_weight (0.3)
       Books by the same author are connected directly in the graph.
       A cold book with 5 ratings inherits signal from popular books
       by the same author through a direct graph path — not just
       through shared author embedding lookup.
       Only connects books in the BOOK node space (offset by n_users).
    """
    print('Building fully augmented adjacency (3 edge types)...')
    N = n_users + n_books

    # Edge type 1: explicit user-book (rating-weighted)
    e_u = train_df['user_idx'].values
    e_b = train_df['book_idx'].values + n_users
    e_w = (train_df['rating'].values / 10.0).astype(np.float32)

    # Edge type 2: implicit user-book
    i_u = implicit_df['user_idx'].values
    i_b = implicit_df['book_idx'].values + n_users
    i_w = np.full(len(i_u), implicit_weight, dtype=np.float32)

    # Edge type 3: author-book
    # For each author with multiple books, connect all book pairs
    author_rows, author_cols, author_vals = [], [], []
    for author_enc, group in books_df.groupby('author_idx'):
        book_idxs = group['book_idx'].values
        if len(book_idxs) < 2:
            continue
        for i in range(len(book_idxs)):
            for j in range(i+1, len(book_idxs)):
                bi = int(book_idxs[i]) + n_users
                bj = int(book_idxs[j]) + n_users
                author_rows.extend([bi, bj])
                author_cols.extend([bj, bi])
                author_vals.extend([author_edge_weight, author_edge_weight])

    author_rows = np.array(author_rows, dtype=np.int64)
    author_cols = np.array(author_cols, dtype=np.int64)
    author_vals = np.array(author_vals, dtype=np.float32)
    print(f'  Author-book edges: {len(author_rows)//2:,} pairs from {books_df["author_idx"].nunique():,} authors')

    # Combine all edge types (symmetric)
    row  = np.concatenate([e_u, e_b, i_u, i_b, author_rows])
    col  = np.concatenate([e_b, e_u, i_b, i_u, author_cols])
    data = np.concatenate([e_w, e_w, i_w, i_w, author_vals])

    A          = csr_matrix((data,(row,col)), shape=(N,N))
    degrees    = np.array(A.sum(axis=1)).flatten()
    d_inv_sqrt = np.where(degrees>0, degrees**-0.5, 0.0)
    D_inv_sqrt = diags(d_inv_sqrt)
    A_hat      = (D_inv_sqrt @ A @ D_inv_sqrt).tocsr().astype(np.float32)

    crow = torch.from_numpy(A_hat.indptr.astype(np.int64))
    col_ = torch.from_numpy(A_hat.indices.astype(np.int64))
    vals = torch.from_numpy(A_hat.data)
    adj  = torch.sparse_csr_tensor(crow, col_, vals, size=(N,N), dtype=torch.float32)

    print(f'  Shape: {N}x{N} | Non-zeros: {A_hat.nnz:,}')
    print(f'  Explicit: {len(e_u):,} | Implicit: {len(i_u):,} | Author-book: {len(author_rows):,}')
    return adj.cpu()

adj = build_norm_adjacency(train_df, implicit_df, meta['books_df'],
                           n_users, n_books, IMPLICIT_WEIGHT, AUTHOR_EDGE_WEIGHT)
user_feat_t = torch.LongTensor(user_feat).to(DEVICE)
book_feat_t = torch.LongTensor(book_feat).to(DEVICE)

cold_user_mask = torch.zeros(n_users, dtype=torch.bool)
for uid in cold_users: cold_user_mask[uid] = True
cold_user_mask = cold_user_mask.to(DEVICE)
print(f'Cold user mask: {cold_user_mask.sum().item():,} flagged')


## Step 7 — LightGCN+ v4 Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class LightGCNPlusV4(nn.Module):
    """
    LightGCN+ v4 — all architectural improvements:

    1. Residual skip connections:
       E^(k) = A_hat @ E^(k-1) + alpha * E^(0)
       Prevents over-smoothing — embeddings always pulled back toward
       the base representation. Critical at 4 layers on sparse data.
       Alpha is a fixed hyperparameter (not learned) for stability.

    2. Adaptive layer dropout:
       Dropout rate increases linearly with layer depth.
       Layer 0: dropout=0.1, Layer 1: 0.11, ..., Layer 4: 0.14
       Deeper layers aggregate from more hops = noisier = more dropout.

    3. Separate learnable temperatures for cold vs warm users:
       Cold embeddings (feature-only) have different magnitude distribution
       than propagated warm embeddings. Shared temperature is a mismatch.
       self.cold_temp and self.warm_temp are learned independently.

    4. L2 normalized scoring (cosine similarity):
       Normalize user and book embeddings to unit sphere before dot product.
       Removes magnitude bias — cold users with smaller embeddings don't
       automatically score lower than warm users.
       This directly fixes the score calibration issue (pos < neg for 83%).
    """

    def __init__(self, n_users, n_books, meta, cold_user_mask,
                 embedding_dim=256, n_layers=4, dropout=0.1,
                 residual_alpha=0.2):
        super().__init__()
        self.n_users        = n_users
        self.n_books        = n_books
        self.n_layers       = n_layers
        self.residual_alpha = residual_alpha
        self.cold_user_mask = cold_user_mask

        # User features: age(16) + country(32) + region(32) = 80
        self.age_emb     = nn.Embedding(meta['n_age_bins'], 16)
        self.country_emb = nn.Embedding(meta['n_countries']+1, 32)
        self.region_emb  = nn.Embedding(meta['n_regions']+1, 32)

        # Book features: author(64) + publisher(32) + decade(16) = 112
        self.author_emb  = nn.Embedding(meta['n_authors']+1, 64)
        self.pub_emb     = nn.Embedding(meta['n_pubs']+1, 32)
        self.decade_emb  = nn.Embedding(meta['n_decades'], 16)

        # Feature projectors
        self.user_proj = nn.Sequential(
            nn.Linear(80, embedding_dim),
            nn.LayerNorm(embedding_dim),
            nn.GELU())
        self.book_proj = nn.Sequential(
            nn.Linear(112, embedding_dim),
            nn.LayerNorm(embedding_dim),
            nn.GELU())

        # CF residual embeddings
        self.user_cf = nn.Embedding(n_users, embedding_dim)
        self.book_cf = nn.Embedding(n_books, embedding_dim)
        nn.init.xavier_uniform_(self.user_cf.weight)
        nn.init.xavier_uniform_(self.book_cf.weight)

        # Learnable layer aggregation weights
        self.layer_weights = nn.Parameter(
            torch.ones(n_layers+1) / (n_layers+1))

        # Separate temperatures for cold vs warm (Improvement 3)
        self.cold_temp = nn.Parameter(torch.tensor(0.5))
        self.warm_temp = nn.Parameter(torch.tensor(0.07))

        # Base dropout — adaptive per layer applied in propagate()
        self.base_dropout = dropout
        self.dropout      = nn.Dropout(p=dropout)

    def get_base_embeddings(self, uf, bf):
        E_users = self.user_proj(torch.cat([
            self.age_emb(uf[:,0]),
            self.country_emb(uf[:,1]),
            self.region_emb(uf[:,2])], dim=1)) + self.user_cf.weight

        E_books = self.book_proj(torch.cat([
            self.author_emb(bf[:,0]),
            self.pub_emb(bf[:,1]),
            self.decade_emb(bf[:,2])], dim=1)) + self.book_cf.weight

        return torch.cat([E_users, E_books], dim=0)

    def propagate(self, adj, uf, bf):
        """
        Propagation with residual connections and adaptive dropout.

        E^(k) = A_hat @ E^(k-1) + alpha * E^(0)

        The residual term alpha * E^(0) is added at each layer.
        This means even at layer 4, every node retains 20% of its
        original feature-based representation — preventing the
        over-smoothing where all embeddings converge to the same vector.
        """
        E0 = self.get_base_embeddings(uf, bf)
        if self.training:
            E0 = self.dropout(E0)

        cold_base = E0[:self.n_users][self.cold_user_mask].clone()

        E_cpu  = E0.cpu().float()
        E0_cpu = E_cpu.clone()   # kept for residual
        layers = [E0]

        for layer_idx in range(self.n_layers):
            # Adaptive dropout: increases with layer depth
            adaptive_p = self.base_dropout * (1 + layer_idx * 0.1)
            adaptive_drop = nn.Dropout(p=min(adaptive_p, 0.5))

            # Propagate + residual skip connection
            E_cpu = torch.sparse.mm(adj, E_cpu) + self.residual_alpha * E0_cpu

            # Apply adaptive dropout to propagated layer (training only)
            E_layer = E_cpu.to(E0.device)
            if self.training:
                E_layer = adaptive_drop(E_layer)
            layers.append(E_layer)

        # Weighted aggregation
        weights = torch.softmax(self.layer_weights, dim=0)
        E_stack = torch.stack(layers, dim=0)
        E_final = (weights.view(-1,1,1) * E_stack).sum(dim=0)

        # Cold start: restore feature-only embeddings for cold users
        E_final[:self.n_users][self.cold_user_mask] = cold_base

        # L2 normalize all embeddings (Improvement 4)
        # Cosine similarity — removes magnitude bias
        E_final = F.normalize(E_final, p=2, dim=1)

        return E_final

    def _get_temperature(self, users):
        """
        Per-user temperature: cold users get cold_temp, warm get warm_temp.
        Clamped to [0.01, 2.0] to prevent degenerate values.
        """
        cold_mask = self.cold_user_mask[users]
        temp = torch.where(
            cold_mask,
            self.cold_temp.clamp(0.01, 2.0).expand_as(cold_mask.float()),
            self.warm_temp.clamp(0.01, 2.0).expand_as(cold_mask.float())
        ).float()
        return temp

    def forward(self, adj, uf, bf, users, pos_books, neg_books):
        E     = self.propagate(adj, uf, bf)
        e_u   = E[users]
        e_pos = E[self.n_users + pos_books]
        e_neg = E[self.n_users + neg_books]

        temp = self._get_temperature(users)
        pos_s = (e_u * e_pos).sum(dim=1) / temp
        neg_s = (e_u * e_neg).sum(dim=1) / temp

        reg = (self.user_cf.weight[users].norm(2).pow(2) +
               self.book_cf.weight[pos_books].norm(2).pow(2) +
               self.book_cf.weight[neg_books].norm(2).pow(2)) / len(users)
        return pos_s, neg_s, reg

    @torch.no_grad()
    def get_all_embeddings(self, adj, uf, bf):
        self.eval()
        return self.propagate(adj, uf, bf)

    @torch.no_grad()
    def score_users(self, E_all, user_indices):
        return torch.matmul(E_all[user_indices], E_all[self.n_users:].T)

model = LightGCNPlusV4(
    n_users, n_books, meta, cold_user_mask,
    embedding_dim=EMBEDDING_DIM, n_layers=N_LAYERS,
    dropout=DROPOUT, residual_alpha=RESIDUAL_ALPHA).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: LightGCN+ v4 | Params: {n_params:,}')
print(f'  Residual alpha: {RESIDUAL_ALPHA}')
print(f'  Cold temp (init): {model.cold_temp.item():.3f}')
print(f'  Warm temp (init): {model.warm_temp.item():.3f}')


## Step 8 — BPR Dataset with hard negative mining

In [ ]:
from torch.utils.data import Dataset, DataLoader

class BPRDataset(Dataset):
    """
    Two-phase negative sampling:

    Phase 1 (epochs 1-HARD_NEG_START): popularity-weighted sampling
      p(neg) ∝ count^0.75 — same as v3
      Focuses on popular books as hard negatives early in training.

    Phase 2 (epochs > HARD_NEG_START): semi-hard negative mining
      For each positive, find negatives that score HIGHER than average
      but LOWER than the positive item.
      These are the most informative negatives — not too easy, not too hard.
      Requires model scores so we pass the model in for phase 2.

    Rating-weighted BPR loss (Improvement 6):
      Each triplet is weighted by the explicit rating value.
      A (user, book_rated_10, neg) triplet contributes 10x more to loss
      than (user, book_rated_1, neg). Consistent with rating-weighted graph.
    """

    def __init__(self, df, n_books):
        self.users      = df['user_idx'].values.astype(np.int64)
        self.books      = df['book_idx'].values.astype(np.int64)
        self.ratings    = df['rating'].values.astype(np.float32)
        self.n_books    = n_books
        self.n          = len(self.users)
        self.hard_mode  = False   # toggled after HARD_NEG_START epochs
        self.model_ref  = None    # set externally when hard mode starts

        self.user_books = defaultdict(set)
        for u,b in zip(self.users, self.books):
            self.user_books[u].add(int(b))

        # Popularity distribution for phase 1
        counts = np.bincount(self.books, minlength=n_books).astype(np.float64)
        counts = np.power(counts+1, 0.75)
        self.neg_probs = (counts/counts.sum()).astype(np.float64)

        self._presample_popularity()

    def _presample_popularity(self):
        """Phase 1: popularity-weighted sampling (vectorized)."""
        candidates  = np.random.choice(self.n_books, size=(self.n,5), p=self.neg_probs)
        neg_samples = np.full(self.n, -1, dtype=np.int64)
        for c_idx in range(5):
            unfilled = neg_samples == -1
            if not unfilled.any(): break
            c     = candidates[unfilled, c_idx]
            u     = self.users[unfilled]
            valid = np.array([c[i] not in self.user_books[u[i]]
                              for i in range(len(u))], dtype=bool)
            neg_samples[np.where(unfilled)[0][valid]] = c[valid]
        for i in np.where(neg_samples == -1)[0]:
            neg = np.random.randint(self.n_books)
            while neg in self.user_books[self.users[i]]:
                neg = np.random.randint(self.n_books)
            neg_samples[i] = neg
        self.neg_samples = neg_samples

    def _presample_hard(self, adj, uf, bf, device):
        """
        Phase 2: semi-hard negative mining.
        For each user, sample 10 candidate negatives, score them with
        the current model, pick the one that scores highest below
        the positive item score. Falls back to popularity sampling
        if no semi-hard negative found.
        """
        if self.model_ref is None:
            self._presample_popularity()
            return

        self.model_ref.eval()
        neg_samples = np.zeros(self.n, dtype=np.int64)

        # Process in batches for efficiency
        batch_size = 512
        with torch.no_grad():
            E_all = self.model_ref.get_all_embeddings(adj, uf, bf)

        for start in range(0, self.n, batch_size):
            end = min(start + batch_size, self.n)
            batch_users = self.users[start:end]
            batch_books = self.books[start:end]

            for i, (uid, pos_bid) in enumerate(zip(batch_users, batch_books)):
                # Sample 10 candidate negatives
                candidates = []
                while len(candidates) < 10:
                    c = np.random.choice(self.n_books, p=self.neg_probs)
                    if c not in self.user_books[uid]:
                        candidates.append(c)

                # Score candidates
                u_t = torch.tensor([uid], dtype=torch.long).to(device)
                c_t = torch.tensor(candidates, dtype=torch.long).to(device)
                e_u = E_all[u_t]
                e_c = E_all[self.model_ref.n_users + c_t]
                scores = (e_u * e_c).sum(dim=1).cpu().numpy()

                # Score positive
                p_t = torch.tensor([pos_bid], dtype=torch.long).to(device)
                e_p = E_all[self.model_ref.n_users + p_t]
                pos_score = (e_u * e_p).sum(dim=1).cpu().numpy()[0]

                # Semi-hard: highest scoring negative that scores below positive
                valid_mask = scores < pos_score
                if valid_mask.any():
                    best_neg = candidates[scores[valid_mask].argmax()]
                else:
                    best_neg = candidates[scores.argmax()]  # fallback

                neg_samples[start + i] = best_neg

        self.neg_samples = neg_samples
        self.model_ref.train()

    def enable_hard_mining(self, model):
        self.hard_mode = True
        self.model_ref = model

    def presample(self, adj=None, uf=None, bf=None, device=None):
        if self.hard_mode and adj is not None:
            self._presample_hard(adj, uf, bf, device)
        else:
            self._presample_popularity()

    def __len__(self): return self.n
    def __getitem__(self, idx):
        return (self.users[idx], self.books[idx],
                self.neg_samples[idx], self.ratings[idx])

def bpr_loss_weighted(pos_scores, neg_scores, ratings, reg, reg_lambda):
    """
    Rating-weighted BPR loss.
    Each triplet weighted by normalized rating (1-10 → 0.1-1.0).
    A strongly liked book (rating=10) penalizes the model more when
    ranked below a negative than a weakly liked book (rating=1).
    Consistent with rating-weighted graph edges.
    """
    weights = (ratings / 10.0).clamp(0.1, 1.0)
    loss    = -(weights * torch.log(
        torch.sigmoid(pos_scores - neg_scores) + 1e-8)).mean()
    return loss + reg_lambda * reg

dataset = BPRDataset(train_df, n_books)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f'Training batches per epoch: {len(loader):,}')
print(f'Hard negative mining starts at epoch: {HARD_NEG_START}')


## Step 9 — Evaluation (sampled + full ranking)

In [ ]:
def build_eval_samples(eval_df, all_pos, n_books, n_neg=99, seed=42):
    rng = np.random.default_rng(seed)
    samples = []
    for row in eval_df.itertuples():
        uid, pos_iid = row.user_idx, row.book_idx
        neg_pool   = [i for i in range(n_books) if i not in all_pos[uid]]
        neg_sample = rng.choice(neg_pool, size=min(n_neg,len(neg_pool)), replace=False).tolist()
        samples.append((uid, pos_iid, neg_sample))
    return samples

@torch.no_grad()
def evaluate_sampled(model, adj, uf, bf, eval_samples, ks, device):
    """99-sampled-negative evaluation — comparable to NeuMF."""
    model.eval()
    E_all   = model.get_all_embeddings(adj, uf, bf)
    results = {k: {'hr':[], 'ndcg':[], 'rank':[]} for k in ks}
    for uid, pos_iid, neg_iids in eval_samples:
        candidates = [pos_iid] + neg_iids
        u_t  = torch.tensor([uid], dtype=torch.long).to(device)
        c_t  = torch.tensor(candidates, dtype=torch.long).to(device)
        scores = (E_all[u_t] * E_all[model.n_users + c_t]).sum(dim=1).cpu().numpy()
        rank = int((scores > scores[0]).sum())
        for k in ks:
            results[k]['hr'].append(1 if rank < k else 0)
            results[k]['ndcg'].append(float(1/np.log2(rank+2)) if rank < k else 0.0)
            results[k]['rank'].append(rank)
    return {k: {'HR@'+str(k): float(np.mean(v['hr'])),
                'NDCG@'+str(k): float(np.mean(v['ndcg']))}
            for k,v in results.items()}

@torch.no_grad()
def evaluate_full_ranking(model, adj, uf, bf, eval_df, train_mask, ks, device, batch_size=256):
    """
    Full ranking evaluation — honest assessment.
    Each user ranks their test item against ALL books minus training items.
    Lower absolute numbers than sampled evaluation but more realistic.
    """
    model.eval()
    E_all      = model.get_all_embeddings(adj, uf, bf)
    test_items = defaultdict(list)
    for u,b in zip(eval_df['user_idx'].values, eval_df['book_idx'].values):
        test_items[u].append(int(b))
    test_users = sorted(test_items.keys())
    results    = {k: {'recall':[], 'ndcg':[], 'hit':[]} for k in ks}
    max_k      = max(ks)

    for start in range(0, len(test_users), batch_size):
        batch_u = test_users[start:start+batch_size]
        u_t     = torch.tensor(batch_u, dtype=torch.long).to(device)
        scores  = model.score_users(E_all, u_t)   # (B, n_books)

        for i, u in enumerate(batch_u):
            tr = list(train_mask[u])
            if tr: scores[i, tr] = -1e9
            topk = scores[i].argsort(descending=True)[:max_k].cpu().numpy()
            gt   = set(test_items[u])
            n_gt = len(gt)
            for k in ks:
                top_k  = topk[:k]
                hits   = np.array([1 if item in gt else 0 for item in top_k])
                n_hits = hits.sum()
                recall = n_hits / min(n_gt, k)
                positions = np.arange(1, k+1)
                dcg   = (hits / np.log2(positions+1)).sum()
                ideal = (1.0 / np.log2(np.arange(1, min(n_gt,k)+1)+1)).sum()
                ndcg  = dcg/ideal if ideal > 0 else 0.0
                results[k]['recall'].append(recall)
                results[k]['ndcg'].append(ndcg)
                results[k]['hit'].append(1 if n_hits > 0 else 0)

    return {k: {'Recall@'+str(k): float(np.mean(v['recall'])),
                'NDCG@'+str(k):   float(np.mean(v['ndcg'])),
                'Hit@'+str(k):    float(np.mean(v['hit']))}
            for k,v in results.items()}

@torch.no_grad()
def evaluate_by_segment(model, adj, uf, bf, eval_samples, cold_users, warm_users, ks, device):
    model.eval()
    E_all = model.get_all_embeddings(adj, uf, bf)
    seg   = {'cold':{k:{'hr':[],'ndcg':[]} for k in ks},
             'warm':{k:{'hr':[],'ndcg':[]} for k in ks}}
    for uid, pos_iid, neg_iids in eval_samples:
        s = 'cold' if uid in cold_users else 'warm'
        candidates = [pos_iid] + neg_iids
        u_t  = torch.tensor([uid], dtype=torch.long).to(device)
        c_t  = torch.tensor(candidates, dtype=torch.long).to(device)
        scores = (E_all[u_t] * E_all[model.n_users + c_t]).sum(dim=1).cpu().numpy()
        rank = int((scores > scores[0]).sum())
        for k in ks:
            seg[s][k]['hr'].append(1 if rank < k else 0)
            seg[s][k]['ndcg'].append(float(1/np.log2(rank+2)) if rank < k else 0.0)
    return {s:{k:{'HR@'+str(k):float(np.mean(v['hr'])),
                  'NDCG@'+str(k):float(np.mean(v['ndcg'])),
                  'n':len(v['hr'])}
               for k,v in res.items()}
            for s,res in seg.items()}

print('Building eval sample sets...')
val_samples  = build_eval_samples(val_df,  all_pos, n_books, N_NEG_EVAL, RANDOM_SEED)
test_samples = build_eval_samples(test_df, all_pos, n_books, N_NEG_EVAL, RANDOM_SEED)
print(f'  Val: {len(val_samples):,} | Test: {len(test_samples):,}')


## Step 10 — Train

In [ ]:
import torch.optim as optim
import time, json

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=0)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-5)

best_ndcg, best_epoch, no_improve = 0.0, 0, 0
best_state       = None
training_history = []
hard_mining_on   = False
t0 = time.time()

print(f'\n{"─"*62}')
print(f'  Training LightGCN+ v4 | {N_EPOCHS} epochs | patience={PATIENCE}')
print(f'{"─"*62}')

for epoch in range(1, N_EPOCHS+1):

    # Switch to hard negative mining after HARD_NEG_START epochs
    if epoch == HARD_NEG_START + 1 and not hard_mining_on:
        dataset.enable_hard_mining(model)
        hard_mining_on = True
        print(f'  ── Switching to semi-hard negative mining (epoch {epoch}) ──')

    # Resample negatives
    if epoch > 1:
        dataset.presample(adj=adj, uf=user_feat_t, bf=book_feat_t, device=DEVICE)

    model.train()
    epoch_loss = 0.0
    t_ep = time.time()

    for users, pos_books, neg_books, ratings in loader:
        users     = users.to(DEVICE)
        pos_books = pos_books.to(DEVICE)
        neg_books = neg_books.to(DEVICE)
        ratings   = ratings.to(DEVICE)

        pos_s, neg_s, reg = model(adj, user_feat_t, book_feat_t,
                                  users, pos_books, neg_books)
        loss = bpr_loss_weighted(pos_s, neg_s, ratings, reg, REG_LAMBDA)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()
    avg_loss = epoch_loss / len(loader)
    elapsed  = time.time() - t_ep
    mode     = 'hard' if hard_mining_on else 'pop'
    print(f'  Ep {epoch:3d}/{N_EPOCHS} | Loss:{avg_loss:.4f} | '
          f'LR:{scheduler.get_last_lr()[0]:.2e} | neg:{mode} | {elapsed:.1f}s')

    if epoch % EVAL_EVERY == 0:
        val_m    = evaluate_sampled(model, adj, user_feat_t, book_feat_t,
                                    val_samples, TOP_K, DEVICE)
        flat_val = {k:v for d in val_m.values() for k,v in d.items()}
        cold_t   = model.cold_temp.item()
        warm_t   = model.warm_temp.item()
        print(f'  Val → ' + '  '.join(f'{k}:{v:.4f}' for k,v in flat_val.items()))
        print(f'  Temps → cold:{cold_t:.4f}  warm:{warm_t:.4f}')
        training_history.append({'epoch':epoch,'loss':avg_loss,
                                  'cold_temp':cold_t,'warm_temp':warm_t,**flat_val})

        val_ndcg = flat_val.get('NDCG@20', flat_val.get('NDCG@10', 0.0))
        if val_ndcg > best_ndcg:
            best_ndcg, best_epoch, no_improve = val_ndcg, epoch, 0
            best_state = {pn:v.clone() for pn,v in model.state_dict().items()}
            torch.save(best_state, os.path.join(RESULTS_DIR,'lightgcn_v4_best.pt'))
            print(f'  ✓ Best NDCG@20: {best_ndcg:.4f} — saved')
        else:
            no_improve += EVAL_EVERY
            print(f'  No improvement for {no_improve} epochs '
                  f'(best {best_ndcg:.4f} @ ep {best_epoch})')
            if no_improve >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

print(f'\nDone in {(time.time()-t0)/60:.1f} min')


## Step 11 — Final evaluation

In [ ]:
if best_state:
    model.load_state_dict(best_state)
    print(f'Loaded best model from epoch {best_epoch}')

# Sampled evaluation (comparable to NeuMF)
test_m    = evaluate_sampled(model, adj, user_feat_t, book_feat_t,
                              test_samples, TOP_K, DEVICE)
flat_test = {k:v for d in test_m.values() for k,v in d.items()}

# Full ranking evaluation (more honest)
full_m    = evaluate_full_ranking(model, adj, user_feat_t, book_feat_t,
                                   test_df, train_mask, TOP_K, DEVICE)
flat_full = {k:v for d in full_m.values() for k,v in d.items()}

# Segment breakdown
seg_m = evaluate_by_segment(model, adj, user_feat_t, book_feat_t,
                             test_samples, cold_users, warm_users, TOP_K, DEVICE)

print('\n' + '='*65)
print('  FINAL RESULTS — LightGCN+ v4')
print('='*65)
print('  SAMPLED (99 neg, comparable to NeuMF):')
for k,v in flat_test.items(): print(f'    {k:<14}: {v:.4f}')
print('\n  FULL RANKING (honest):')
for k,v in flat_full.items(): print(f'    {k:<14}: {v:.4f}')

print('\n  COLD vs WARM (sampled):')
print(f'  {"Metric":<12} {"Cold":>12} {"Warm":>12}')
print(f'  {"-"*38}')
for k in TOP_K:
    print(f'  HR@{k:<9} {seg_m["cold"][k]["HR@"+str(k)]:>12.4f} {seg_m["warm"][k]["HR@"+str(k)]:>12.4f}')
    print(f'  NDCG@{k:<7} {seg_m["cold"][k]["NDCG@"+str(k)]:>12.4f} {seg_m["warm"][k]["NDCG@"+str(k)]:>12.4f}')
print(f'  n          {seg_m["cold"][10]["n"]:>12,} {seg_m["warm"][10]["n"]:>12,}')

print('\n  vs v3:')
v3 = {'HR@5':0.3601,'HR@10':0.4636,'HR@20':0.5998,
      'NDCG@5':0.2677,'NDCG@10':0.3011,'NDCG@20':0.3354}
print(f'  {"Metric":<12} {"v3":>10} {"v4":>10} {"Δ":>8}')
print(f'  {"-"*44}')
for k,v3v in v3.items():
    v4v = flat_test.get(k,0)
    d   = v4v - v3v
    print(f'  {k:<12} {v3v:>10.4f} {v4v:>10.4f} {d:>+8.4f} {"✓" if d>0 else "✗"}')
print('='*65)

# Learned temperatures
cold_t = model.cold_temp.item()
warm_t = model.warm_temp.item()
print(f'\n  Learned temperatures:')
print(f'    Cold users: {cold_t:.4f} (init: 0.5000)')
print(f'    Warm users: {warm_t:.4f} (init: 0.0700)')

results = {
    'sampled_test': flat_test,
    'full_ranking_test': flat_full,
    'cold_metrics': {str(k):seg_m['cold'][k] for k in TOP_K},
    'warm_metrics': {str(k):seg_m['warm'][k] for k in TOP_K},
    'training_history': training_history,
    'best_epoch': best_epoch,
    'learned_cold_temp': cold_t,
    'learned_warm_temp': warm_t,
    'improvements': ['rating_weighted_edges','implicit_edges','author_book_edges',
                     'region_features','residual_connections','adaptive_dropout',
                     'separate_temperatures','cosine_similarity','weighted_bpr',
                     'hard_negative_mining','full_ranking_eval'],
}
with open(os.path.join(RESULTS_DIR,'results.json'),'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved to {RESULTS_DIR}/results.json')


## Step 12 — Diagnostic plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

SAVE_DIR = os.path.join(RESULTS_DIR, 'error_analysis')
os.makedirs(SAVE_DIR, exist_ok=True)

def save(fig, name):
    fig.savefig(os.path.join(SAVE_DIR,name), dpi=150, bbox_inches='tight')
    plt.close(fig); print(f'  ✓ {name}')

def _activity_label(n):
    if n<=10:    return 'Cold (≤10)'
    elif n<=30:  return 'Warm (11-30)'
    elif n<=100: return 'Active (31-100)'
    else:        return 'Power (>100)'

model.eval()
E_all = model.get_all_embeddings(adj, user_feat_t, book_feat_t)
pop   = np.zeros(n_books, dtype=np.int32)
for b in train_df['book_idx'].values: pop[b] += 1
user_activity = train_df.groupby('user_idx')['book_idx'].count().to_dict()

records = []
for uid, pos_iid, neg_iids in test_samples:
    candidates = [pos_iid] + neg_iids
    u_t  = torch.tensor([uid], dtype=torch.long).to(DEVICE)
    c_t  = torch.tensor(candidates, dtype=torch.long).to(DEVICE)
    scores = (E_all[u_t] * E_all[model.n_users + c_t]).sum(dim=1).cpu().numpy()
    rank   = int((scores > scores[0]).sum())
    records.append({
        'uid':uid,'rank':rank,'pos_score':float(scores[0]),
        'max_neg_score':float(scores[1:].max()),
        'n_train_items':int(user_activity.get(uid,0)),
        'item_popularity':int(pop[pos_iid]),'is_cold':uid in cold_users,
        'hit@5':int(rank<5),'hit@10':int(rank<10),'hit@20':int(rank<20),
        'ndcg@5':float(1/np.log2(rank+2)) if rank<5 else 0.0,
        'ndcg@10':float(1/np.log2(rank+2)) if rank<10 else 0.0,
        'ndcg@20':float(1/np.log2(rank+2)) if rank<20 else 0.0,
    })

diag = pd.DataFrame(records)
diag['activity_segment'] = diag['n_train_items'].apply(_activity_label)
diag['pop_bin'] = pd.qcut(diag['item_popularity'].clip(lower=1), q=4,
    labels=['Q1 (cold)','Q2','Q3','Q4 (popular)'])
cold_diag = diag[diag['is_cold']]
warm_diag = diag[~diag['is_cold']]
print(f'Diagnostics: {len(diag):,} | Cold: {len(cold_diag):,} | Warm: {len(warm_diag):,}')
print(f'  Mean pos score:     {diag.pos_score.mean():.4f}')
print(f'  Mean max neg score: {diag.max_neg_score.mean():.4f}')
print(f'  % pos > max neg:    {(diag.pos_score > diag.max_neg_score).mean()*100:.1f}%')
print(f'  Median rank:        {diag["rank"].median():.0f}')

order_act = ['Cold (≤10)','Warm (11-30)','Active (31-100)','Power (>100)']
order_act = [o for o in order_act if o in diag['activity_segment'].unique()]

# Plot 1: Rank CDF
ranks = np.sort(diag['rank'].values)
cdf   = np.arange(1,len(ranks)+1)/len(ranks)
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(ranks, cdf, color='steelblue', lw=2)
for k,c in zip([5,10,20],['#e74c3c','#e67e22','#2ecc71']):
    ax.axvline(k-0.5,color=c,ls='--',lw=1.2,label=f'HR@{k}={(ranks<k).mean():.3f}')
ax.set_xlabel('Rank'); ax.set_ylabel('CDF'); ax.set_title('Rank CDF — LightGCN+ v4')
ax.legend(); ax.set_xlim(0,100)
save(fig,'01_rank_cdf.png')

# Plot 2: Cold vs Warm
fig, axes = plt.subplots(1,2,figsize=(12,5))
x = np.arange(3); w = 0.35
axes[0].bar(x-w/2,[cold_diag[f'hit@{k}'].mean() for k in [5,10,20]],w,
    label=f'Cold (n={len(cold_diag):,})',color='#e74c3c',alpha=0.8)
axes[0].bar(x+w/2,[warm_diag[f'hit@{k}'].mean() for k in [5,10,20]],w,
    label=f'Warm (n={len(warm_diag):,})',color='#2ecc71',alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(['HR@5','HR@10','HR@20'])
axes[0].set_title('Cold vs Warm — Hit Rate'); axes[0].legend(); axes[0].set_ylim(0,1)
axes[1].bar(x-w/2,[cold_diag[f'ndcg@{k}'].mean() for k in [5,10,20]],w,
    label=f'Cold (n={len(cold_diag):,})',color='#e74c3c',alpha=0.8)
axes[1].bar(x+w/2,[warm_diag[f'ndcg@{k}'].mean() for k in [5,10,20]],w,
    label=f'Warm (n={len(warm_diag):,})',color='#2ecc71',alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(['NDCG@5','NDCG@10','NDCG@20'])
axes[1].set_title('Cold vs Warm — NDCG'); axes[1].legend(); axes[1].set_ylim(0,0.5)
plt.tight_layout(); save(fig,'02_cold_vs_warm.png')

# Plot 3: Activity segments
seg_stats = diag.groupby('activity_segment')[['hit@5','hit@10','hit@20','ndcg@10']].mean().reindex(order_act)
seg_n     = diag.groupby('activity_segment').size().reindex(order_act)
fig, axes = plt.subplots(1,2,figsize=(13,5))
seg_stats[['hit@5','hit@10','hit@20']].plot(kind='bar',ax=axes[0],
    color=['#3498db','#2ecc71','#e67e22'],edgecolor='white')
axes[0].set_title('Hit Rate by Activity Segment')
axes[0].set_xticklabels([f'{s}\n(n={seg_n[s]})' for s in order_act],rotation=0)
seg_stats['ndcg@10'].plot(kind='bar',ax=axes[1],color='#9b59b6',edgecolor='white')
axes[1].set_title('NDCG@10 by Activity Segment')
axes[1].set_xticklabels([f'{s}\n(n={seg_n[s]})' for s in order_act],rotation=0)
plt.tight_layout(); save(fig,'03_activity_segments.png')

# Plot 4: Popularity bias
fig, axes = plt.subplots(1,2,figsize=(12,5))
diag.groupby('pop_bin')['hit@10'].mean().plot(kind='bar',ax=axes[0],color='#3498db',edgecolor='white')
axes[0].set_title('HR@10 by Item Popularity Quartile')
axes[0].set_xticklabels(axes[0].get_xticklabels(),rotation=0)
axes[1].scatter(diag['item_popularity'],diag['rank'],alpha=0.15,s=8,color='coral')
axes[1].set_xlabel('Item popularity'); axes[1].set_ylabel('Rank')
axes[1].set_title('Rank vs Popularity'); axes[1].set_xscale('log')
plt.tight_layout(); save(fig,'04_popularity_bias.png')

# Plot 5: Score calibration
fig, axes = plt.subplots(1,2,figsize=(12,5))
axes[0].hist(diag['pos_score'],bins=40,alpha=0.6,label='Positive',color='#2ecc71')
axes[0].hist(diag['max_neg_score'],bins=40,alpha=0.6,label='Max negative',color='#e74c3c')
axes[0].set_title('Score Distribution'); axes[0].legend()
gap = diag['pos_score'] - diag['max_neg_score']
axes[1].hist(gap[diag['hit@10']==1],bins=30,alpha=0.6,color='#2ecc71',label='Hit')
axes[1].hist(gap[diag['hit@10']==0],bins=30,alpha=0.6,color='#e74c3c',label='Miss')
axes[1].axvline(0,color='black',ls='--')
axes[1].set_title('Score Gap (cosine — L2 normalized)'); axes[1].legend()
plt.tight_layout(); save(fig,'05_score_calibration.png')

# Plot 6: Heatmap
order_pop = [c for c in ['Q1 (cold)','Q2','Q3','Q4 (popular)']
             if c in diag['pop_bin'].cat.categories]
pivot = diag.pivot_table(index='activity_segment',columns='pop_bin',
    values='hit@10',aggfunc='mean').reindex(index=order_act,columns=order_pop)
fig, ax = plt.subplots(figsize=(9,5))
sns.heatmap(pivot,annot=True,fmt='.3f',cmap='RdYlGn',vmin=0,vmax=1,ax=ax)
ax.set_title('HR@10: User Activity × Item Popularity')
plt.tight_layout(); save(fig,'06_heatmap.png')

# Plot 7: Training curve with temperature evolution
if training_history:
    epochs_h  = [h['epoch'] for h in training_history]
    losses_h  = [h['loss']  for h in training_history]
    ndcg_h    = [h.get('NDCG@20',h.get('NDCG@10',0)) for h in training_history]
    cold_t_h  = [h.get('cold_temp',0.5) for h in training_history]
    warm_t_h  = [h.get('warm_temp',0.07) for h in training_history]

    fig, axes = plt.subplots(1,2,figsize=(14,5))
    ax2 = axes[0].twinx()
    axes[0].plot(epochs_h,losses_h,color='#e74c3c',lw=2,label='BPR Loss')
    ax2.plot(epochs_h,ndcg_h,color='#2ecc71',lw=2,label='NDCG@20 (val)')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss',color='#e74c3c')
    ax2.set_ylabel('NDCG@20',color='#2ecc71')
    axes[0].set_title('Training Curve')
    l1,lb1=axes[0].get_legend_handles_labels(); l2,lb2=ax2.get_legend_handles_labels()
    axes[0].legend(l1+l2,lb1+lb2,loc='upper right')

    axes[1].plot(epochs_h,cold_t_h,color='#e74c3c',lw=2,label='Cold temp')
    axes[1].plot(epochs_h,warm_t_h,color='#2ecc71',lw=2,label='Warm temp')
    axes[1].axvline(HARD_NEG_START,color='gray',ls='--',lw=1,label=f'Hard neg start (ep {HARD_NEG_START})')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Temperature')
    axes[1].set_title('Learned Temperatures Over Training')
    axes[1].legend()
    plt.tight_layout(); save(fig,'07_training_curve.png')

# Plot 8: v2 vs v3 vs v4 comparison
all_versions = {
    'v2 (Cold Start)': {'HR@5':0.3545,'HR@10':0.4652,'HR@20':0.6001,
                        'NDCG@5':0.2605,'NDCG@10':0.2962,'NDCG@20':0.3302},
    'v3 (+3 improvements)': {'HR@5':0.3601,'HR@10':0.4636,'HR@20':0.5998,
                             'NDCG@5':0.2677,'NDCG@10':0.3011,'NDCG@20':0.3354},
    'v4 (final)': flat_test,
    'NeuMF': {'HR@5':0.2677,'HR@10':0.4404,'HR@20':0.5904,
              'NDCG@5':0.1572,'NDCG@10':0.2134,'NDCG@20':0.2513},
}
labels  = ['HR@5','HR@10','HR@20','NDCG@5','NDCG@10','NDCG@20']
colors  = ['#3498db','#2ecc71','#e74c3c','#e67e22']
x       = np.arange(len(labels)); w = 0.2
fig, ax = plt.subplots(figsize=(13,6))
for i,(name,vals) in enumerate(all_versions.items()):
    ax.bar(x + (i-1.5)*w, [vals.get(l,0) for l in labels],
           w, label=name, color=colors[i], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_title('Model Comparison: NeuMF vs LightGCN+ versions')
ax.legend(); ax.set_ylim(0, 0.8)
plt.tight_layout(); save(fig,'08_version_comparison.png')

# Plot 9: Full ranking vs sampled evaluation comparison
sampled_vals = [flat_test.get(f'HR@{k}',0) for k in TOP_K] +                [flat_test.get(f'NDCG@{k}',0) for k in TOP_K]
full_vals    = [flat_full.get(f'Hit@{k}',0) for k in TOP_K] +                [flat_full.get(f'NDCG@{k}',0) for k in TOP_K]
xlabels      = [f'HR@{k}' for k in TOP_K] + [f'NDCG@{k}' for k in TOP_K]
x2 = np.arange(len(xlabels)); w2 = 0.35
fig, ax = plt.subplots(figsize=(10,5))
ax.bar(x2-w2/2, sampled_vals, w2, label='Sampled (99 neg)', color='#3498db', alpha=0.85)
ax.bar(x2+w2/2, full_vals,    w2, label='Full ranking',     color='#e74c3c', alpha=0.85)
ax.set_xticks(x2); ax.set_xticklabels(xlabels)
ax.set_title('Sampled vs Full Ranking Evaluation')
ax.legend(); ax.set_ylim(0, 0.8)
plt.tight_layout(); save(fig,'09_sampled_vs_full.png')

print('\nAll plots saved.')


## Step 13 — Summary report

In [ ]:
report = {
    'model': 'LightGCN+ v4',
    'improvements': [
        'rating_weighted_graph_edges',
        'implicit_auxiliary_edges',
        'author_book_graph_edges',
        'region_user_features',
        'residual_skip_connections',
        'adaptive_layer_dropout',
        'separate_cold_warm_temperature',
        'cosine_similarity_scoring',
        'rating_weighted_bpr_loss',
        'semi_hard_negative_mining',
        'full_ranking_evaluation',
    ],
    'cold_threshold': COLD_THRESHOLD,
    'n_cold_test': int(len(cold_diag)),
    'n_warm_test': int(len(warm_diag)),
    'n_instances': int(len(diag)),
    # Sampled metrics
    'HR@5':   round(diag['hit@5'].mean(),4),
    'HR@10':  round(diag['hit@10'].mean(),4),
    'HR@20':  round(diag['hit@20'].mean(),4),
    'NDCG@5': round(diag['ndcg@5'].mean(),4),
    'NDCG@10':round(diag['ndcg@10'].mean(),4),
    'NDCG@20':round(diag['ndcg@20'].mean(),4),
    # Full ranking metrics
    'full_Recall@10': round(flat_full.get('Recall@10',0),4),
    'full_NDCG@10':   round(flat_full.get('NDCG@10',0),4),
    'full_Hit@10':    round(flat_full.get('Hit@10',0),4),
    # Diagnostics
    'median_rank': int(diag['rank'].median()),
    'mean_pos_score': round(diag['pos_score'].mean(),4),
    'mean_max_neg_score': round(diag['max_neg_score'].mean(),4),
    'pct_pos_beats_neg': round((diag['pos_score']>diag['max_neg_score']).mean()*100,1),
    # Cold/warm
    'cold_HR@10':  round(cold_diag['hit@10'].mean(),4),
    'cold_NDCG@10':round(cold_diag['ndcg@10'].mean(),4),
    'warm_HR@10':  round(warm_diag['hit@10'].mean(),4),
    'warm_NDCG@10':round(warm_diag['ndcg@10'].mean(),4),
    # Training
    'best_epoch': best_epoch,
    'learned_cold_temp': round(model.cold_temp.item(),4),
    'learned_warm_temp': round(model.warm_temp.item(),4),
}

print('\n' + '='*55)
print('  FINAL REPORT — LightGCN+ v4')
print('='*55)
for k,v in report.items(): print(f'  {k:<30}: {v}')
print('='*55)

with open(os.path.join(SAVE_DIR,'report.json'),'w') as f:
    json.dump(report, f, indent=2)
diag.to_csv(os.path.join(SAVE_DIR,'diagnostics.csv'), index=False)
print(f'\nSaved to {SAVE_DIR}/')


## Step 14 — Layer weight analysis

In [ ]:
w = torch.softmax(model.layer_weights, dim=0).detach().cpu().numpy()
print('Learned per-layer aggregation weights (with residual connections):')
print()
for i, wi in enumerate(w):
    bar = '█' * int(wi * 60)
    print(f'  Layer {i} ({i}-hop): {wi:.4f}  {bar}')
print()
print(f'Learned cold temperature:  {model.cold_temp.item():.4f}')
print(f'Learned warm temperature:  {model.warm_temp.item():.4f}')
print()
print('Report insights:')
print('  - If layer 0 weight is highest: base features dominate (sparse data)')
print('  - If layers 2-3 dominate: multi-hop graph signal is working')
print('  - If cold_temp >> warm_temp: cold user scores are naturally flatter')
print('  - If cold_temp << warm_temp: cold user scores are sharper than warm')


## Step 15 — Sample recommendations

In [ ]:
import random

def recommend(model, adj, uf, bf, uid, n=10):
    model.eval()
    E_all   = model.get_all_embeddings(adj, uf, bf)
    e_user  = E_all[uid].unsqueeze(0)
    e_books = E_all[model.n_users:]
    scores  = torch.matmul(e_user, e_books.T).squeeze().cpu().numpy()
    for b in train_mask[uid]: scores[b] = -1e9
    top_iids   = np.argsort(-scores)[:n]
    idx2item   = {v:k for k,v in meta['book_map'].items()}
    books_meta = meta['books_meta']
    recs = []
    for rank, iid in enumerate(top_iids, 1):
        isbn = idx2item.get(int(iid),'?')
        try:
            row = books_meta.loc[isbn]
            title  = str(row.get('title','?'))[:45]
            author = str(row.get('author','?'))[:25]
        except: title, author = '?','?'
        recs.append({'rank':rank,'score':float(scores[iid]),
                     'title':title,'author':author})
    return recs

cold_test_users = [uid for uid,_,_ in test_samples if uid in cold_users]
warm_test_users = [uid for uid,_,_ in test_samples if uid in warm_users]

for label, uid in [('COLD', random.choice(cold_test_users)),
                   ('WARM', random.choice(warm_test_users))]:
    n_train = user_activity.get(uid, 0)
    recs    = recommend(model, adj, user_feat_t, book_feat_t, uid)
    print(f'\n{"─"*70}')
    print(f'  {label} user {uid} ({n_train} training ratings)')
    print(f'{"─"*70}')
    for r in recs:
        print(f'  #{r["rank"]:>2}  [{r["score"]:+.4f}]  {r["title"]:<45}  {r["author"]}')
print(f'{"─"*70}')
